# Practical 2 — Data Wrangling (Academic performance)

Objective: create a small `Academic performance` dataset and demonstrate missing-value handling, outlier detection & IQR capping, a transformation, and short reasoning for each step.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

plt.rcParams['figure.figsize'] = (8,4)

## 1) Create dataset

The dataset contains student academic scores and a few related fields. I intentionally include some missing values and an inconsistent entry to demonstrate cleaning.

In [ ]:
data = {
    'StudentID': list(range(1,16)),
    'Math_Score': [75,70,72,68,120,80,60,78,66,55,90,None,85,200,73],
    'Reading_Score': [78,82,None,75,90,95,77,92,88,91,85,80,76,300,79],
    'Writing_Score': [62,68,70,None,74,80,60,76,69,72,65,68,70,85,71],
    'Placement_Score': [80,85,90,85,95,100,76,96,88,93,70,66,82,400,89],
    'Placement_Offer_Count': [1,2,'N/A',3,3,4,0,3,2,3,1,2,1,5,2],
    'Club_Join_Year': [2018,2019,2019,2018,2020,2021,2018,2021,2020,2021,2018,2019,None,2022,2019],
}

df = pd.DataFrame(data)
# coerce numeric columns that may have inconsistent entries
df['Placement_Offer_Count'] = pd.to_numeric(df['Placement_Offer_Count'], errors='coerce')
df.head()

## 2) Scan for missing values and inconsistencies

We'll show missing counts and basic type checks.

In [ ]:
# Missing values per column
print(df.isna().sum())
print('\nData types:')
print(df.dtypes)

# Show rows with any NaN or extreme/inconsistent values (simple filter)
df[df.isna().any(axis=1)]

### Approach (missing/inconsistent):
- `Math_Score`, `Reading_Score`, `Writing_Score`: numeric test scores — use mean/median depending on robustness (median for skewed).
- `Placement_Offer_Count`: small integer count with one inconsistent `N/A` coerced to `NaN` — impute median (robust).
- `Club_Join_Year`: fill with mode (most common year) as a reasonable default.

In [ ]:
# Impute using chosen strategies
df2 = df.copy()
df2['Math_Score'] = df2['Math_Score'].fillna(df2['Math_Score'].mean())
df2['Reading_Score'] = df2['Reading_Score'].fillna(df2['Reading_Score'].median())
df2['Writing_Score'] = df2['Writing_Score'].fillna(df2['Writing_Score'].median())
df2['Placement_Offer_Count'] = df2['Placement_Offer_Count'].fillna(df2['Placement_Offer_Count'].median())
df2['Club_Join_Year'] = df2['Club_Join_Year'].fillna(df2['Club_Join_Year'].mode().iloc[0])
print('\nAfter imputation:')
print(df2.isna().sum())
df2.head()

## 3) Outlier detection (IQR) and IQR capping

We'll detect outliers using the IQR rule and then apply capping (clip to [lower, upper]) to reduce extreme influence while keeping records.

In [ ]:
numeric_cols = ['Math_Score','Reading_Score','Writing_Score','Placement_Score','Placement_Offer_Count']
outlier_summary = {}
for col in numeric_cols:
    Q1 = df2[col].quantile(0.25)
    Q3 = df2[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    mask = (df2[col] < lower) | (df2[col] > upper)
    outlier_summary[col] = {'Q1':Q1, 'Q3':Q3, 'IQR':IQR, 'lower':lower, 'upper':upper, 'n_outliers': int(mask.sum())}
outlier_summary

In [ ]:
# Show rows flagged as outliers in any numeric column
masks = []
for col in numeric_cols:
    Q1 = df2[col].quantile(0.25)
    Q3 = df2[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    masks.append((df2[col] < lower) | (df2[col] > upper))
outlier_mask = np.column_stack(masks).any(axis=1)
df2[outlier_mask]

In [ ]:
# IQR capping: clip values to [lower, upper] computed from df2
df_capped = df2.copy()
for col in numeric_cols:
    Q1 = df2[col].quantile(0.25)
    Q3 = df2[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df_capped[col] = df_capped[col].clip(lower, upper)

# Compare before/after with simple boxplots
fig, axes = plt.subplots(1,2, figsize=(12,4))
df2[numeric_cols].boxplot(ax=axes[0])
axes[0].set_title('Before capping')
df_capped[numeric_cols].boxplot(ax=axes[1])
axes[1].set_title('After IQR capping')
plt.tight_layout()
plt.show()
df_capped.head()

## 4) Transformation

We apply `log1p` (log(1+x)) to `Placement_Offer_Count` to reduce skewness for count data (contains 0), and Min–Max scaling to `Placement_Score` to change its scale to [0,1] for interpretability.

In [ ]:
from scipy import stats
# Skewness before
print('Skewness before:')
print(df_capped[numeric_cols].skew())

# log1p transform for counts
df_trans = df_capped.copy()
df_trans['Placement_Offer_Count_log1p'] = np.log1p(df_trans['Placement_Offer_Count'])
# MinMax scaling for Placement_Score
scaler = MinMaxScaler()
df_trans['Placement_Score_scaled'] = scaler.fit_transform(df_trans[['Placement_Score']])

print('\nSkewness after transforms:')
print(df_trans[['Placement_Offer_Count','Placement_Offer_Count_log1p','Placement_Score','Placement_Score_scaled']].skew())
df_trans.head()

## Reasoning and summary

- Missing values: used mean for `Math_Score` (reasonable for balanced score distribution), median for `Reading`/`Writing` (robust to skew/outliers), and mode for `Club_Join_Year` to keep a plausible year.
- Outliers: detected via IQR rule (1.5*IQR). IQR capping (clipping to bounds) preserves records while reducing influence of extremes — appropriate for small classroom datasets.
- Transformation: `log1p` reduces skewness for count data and handles zeros; Min–Max scaling makes `Placement_Score` easier to compare on a 0–1 scale.

The cleaned `df_trans` contains the final transformed dataset. Save with `df_trans.to_csv('Students_Performance_cleaned.csv', index=False)` if needed.